# Variance-reduction experiments

Monte Carlo comparison of base, antithetic, common-random-number, and control-variate estimators. The default scenario grid is heavy; reduce `scenarios` or `indRepsSims` for a quick check.


In [ ]:
from simulate_paths import NUM_HAZARD_STEPS, simulatePaths
import matplotlib.pyplot as plt
import numpy as np
np.random.seed(0)


scenarios = np.arange(1000,50000,1000)
cases = 3 
resultsPV_IG = np.zeros((cases,len(scenarios)))
resultsSE_IG = np.zeros((cases,len(scenarios)))
resultsPV_HY = np.zeros((cases,len(scenarios)))
resultsSE_HY = np.zeros((cases,len(scenarios)))

pct = 0.1 # 10% of the random variates will be used to estimate the correlation 

indReps = 5 
indRepsSims = 100_000 # 
# indRepsSims = 100
M = NUM_HAZARD_STEPS
resultsIG = np.zeros(indReps)
resultsHY = np.zeros(indReps)

for jx in range(indReps):
    Z = np.random.normal(loc=0.0, scale=1.0,size = (indRepsSims,M))
    PV_paths_IG, PV_paths_HY, _, _ = simulatePaths(Z,useRangeAccrual = True)
    resultsIG[jx] = np.mean(PV_paths_IG)
    resultsHY[jx] = np.mean(PV_paths_HY) 

answerIG = np.mean(resultsIG)
answerHY = np.mean(resultsHY)


In [ ]:

for ix, numOfSims in enumerate(scenarios):
    print(numOfSims)
    M = NUM_HAZARD_STEPS
    Z = np.random.normal(loc=0.0, scale=1.0,size = (numOfSims,M))
    # print(Z.shape)
    
    #base case no variance reduction
    PV_paths_IG, PV_paths_HY, _, _ = simulatePaths(Z,useRangeAccrual = True)

    resultsPV_IG[0,ix] = np.mean(PV_paths_IG)
    resultsSE_IG[0,ix] = np.std(PV_paths_IG) / np.sqrt(len(PV_paths_IG))
    resultsPV_HY[0,ix] = np.mean(PV_paths_HY)
    resultsSE_HY[0,ix] = np.std(PV_paths_HY) / np.sqrt(len(PV_paths_HY))

    # antithetics
    # using half to make it comparable to base case
    PV_paths_IG_1, PV_paths_HY_1, _, _ = simulatePaths( Z[:int(numOfSims/2),:],useRangeAccrual = True)
    PV_paths_IG_2, PV_paths_HY_2, _, _ = simulatePaths(-Z[:int(numOfSims/2),:],useRangeAccrual = True)

    PV_paths_HY = 0.5*(PV_paths_HY_1 + PV_paths_HY_2) 
    PV_paths_IG = 0.5*(PV_paths_IG_1 + PV_paths_IG_2) 

    resultsPV_IG[1,ix] = np.mean(PV_paths_IG)
    resultsSE_IG[1,ix] = np.std(PV_paths_IG) / np.sqrt(len(PV_paths_IG))
    resultsPV_HY[1,ix] = np.mean(PV_paths_HY)
    resultsSE_HY[1,ix] = np.std(PV_paths_HY) / np.sqrt(len(PV_paths_HY))

    # control variates with common random numbers
    # to make it fair, we will use the same number of variables as the other two cases
    index = int(numOfSims*pct)
    Z_beta = Z[:index,:]
    Z_control = Z[index:,:]

    # vanilla case
    PV_paths_IG_van_beta, PV_paths_HY_van_beta, pvAnalyticalVanillaIG, pvAnalyticalVanillaHY = simulatePaths(Z_beta,useRangeAccrual = False)
    # exotic case 
    PV_paths_IG_exo_beta, PV_paths_HY_exo_beta, _, _                                         = simulatePaths(Z_beta,useRangeAccrual = True)

  
    IG_cov = np.cov(PV_paths_IG_exo_beta,PV_paths_IG_van_beta)
    HY_cov = np.cov(PV_paths_HY_exo_beta,PV_paths_HY_van_beta)

    beta_ig = IG_cov[0][1] / IG_cov[1][1]
    beta_hy = HY_cov[0][1] / HY_cov[1][1]

    
    # vanilla case
    PV_paths_IG_van, PV_paths_HY_van, _, _  = simulatePaths(Z_control,useRangeAccrual = False)
    # exotic case 
    PV_paths_IG_exo, PV_paths_HY_exo, _, _  = simulatePaths(Z_control,useRangeAccrual = True)

  
    PV_paths_IG = PV_paths_IG_exo - beta_ig * (PV_paths_IG_van - pvAnalyticalVanillaIG)
    PV_paths_HY = PV_paths_HY_exo - beta_hy * (PV_paths_HY_van - pvAnalyticalVanillaHY)

    resultsPV_IG[2,ix] = np.mean(PV_paths_IG)
    resultsSE_IG[2,ix] = np.std(PV_paths_IG) / np.sqrt(len(PV_paths_IG))
    resultsPV_HY[2,ix] = np.mean(PV_paths_HY)
    resultsSE_HY[2,ix] = np.std(PV_paths_HY) / np.sqrt(len(PV_paths_HY))



In [ ]:
scenarios.shape, answerIG.shape

In [ ]:
from fig_style import use_ieee_style, new_figure, save_figure
from matplotlib.ticker import FormatStrFormatter


use_ieee_style(single_column=False, height_factor=0.9, use_tex=True)

answerIG = answerIG*np.ones_like(scenarios)
answerHY= answerHY*np.ones_like(scenarios)

# should probably get this in scipy stats 
z = 2.576  # 99% confidence interval 

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey='row')

# Add figure-level legend
from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], color='black', lw=1.5, label='True Value'),
    Line2D([0], [0], color='black', lw=1.0, linestyle='--', label='99\% CI'),
    Line2D([0], [0], color='black', marker='o', linestyle='', markersize=6,
           label='Monte Carlo Estimates')
]

fig.legend(
    handles=legend_elements,
    loc='lower center',
    ncol=3,
    frameon=False,
    fontsize=11,
    bbox_to_anchor=(0.5, -0.06)
)


titles = [
    "IG Base Case", "IG Antithetic", "IG Control Variate",
    "HY Base Case", "HY Antithetic", "HY Control Variate"
]

# Loop over the 6 subplots
for j in range(3):  # columns = base, antithetic, control variate
    
    # ---------- IG (top row) ----------
    ax_ig = axes[0, j]
    ax_ig.set_xlim(scenarios.min(), scenarios.max())

    # analytical line
    ax_ig.plot(scenarios, answerIG*np.ones_like(scenarios), linewidth=1.5, color="black")

    # bands = analytical ± std err
    ax_ig.plot(
        scenarios,
        answerIG + z * resultsSE_IG[j, :],
        linestyle="--",
        linewidth=1.0,
        color="black",
    )
    ax_ig.plot(
        scenarios,
        answerIG - z * resultsSE_IG[j, :],
        linestyle="--",
        linewidth=1.0,
        color="black",
    )

    # markers for MC estimates
    ax_ig.plot(
        scenarios,
        resultsPV_IG[j, :],
        marker="o",
        linestyle="",
        ms=6,
        color="black"
    )

    ax_ig.set_title(titles[j])

    # ---------- HY (bottom row) ----------
    ax_hy = axes[1, j]
    ax_hy.set_xlim(scenarios.min(), scenarios.max())

    # analytical line
    ax_hy.plot(scenarios, answerHY*np.ones_like(scenarios),
               linewidth=1.5, color="black")

    # bands
    ax_hy.plot(
        scenarios,
        answerHY + z * resultsSE_HY[j, :],
        linestyle="--",
        linewidth=1.0,
        color="black",
    )
    ax_hy.plot(
        scenarios,
        answerHY - z * resultsSE_HY[j, :],
        linestyle="--",
        linewidth=1.0,
        color="black",
    )

    # markers
    ax_hy.plot(
        scenarios,
        resultsPV_HY[j, :],
        marker="o",
        linestyle="",
        ms=6,
        color="black"
    )

    ax_hy.set_title(titles[3 + j])

# Label axes
for ax in axes[1, :]:
    ax.set_xlabel("Number of simulations")

axes[0, 0].set_ylabel("Present Value (IG)")
axes[1, 0].set_ylabel("Present Value (HY)")

plt.tight_layout()
plt.show()
